# Análisis Exploratorio: Desempeño de Estudiantes en Educación Secundaria
## Exploración de hábitos de estudio, contexto familiar y rendimiento académico

**Equipo:** Alexandra Guaman  
**Fecha:** Noviembre 2026  
**Conjunto de datos:** [Student Performance Dataset](https://www.kaggle.com/datasets/larsen0966/student-performance-data-set) (Kaggle, `larsen0966` — Basado en UCI Machine Learning Repository)

---

### Pregunta de investigación

> **¿Qué factores (tiempo de estudio semanal, apoyo educativo familiar, acceso a internet, actividades extracurriculares y educación materna) se asocian con un mayor puntaje final (G3), y qué tan fuerte es cada asociación?**

### Objetivos
1. Describir la estructura del conjunto de datos: observaciones, variables y tipos.
2. Evaluar la calidad de los datos: valores nulos, duplicados, consistencia, rangos y valores atípicos.
3. Resumir las calificaciones mediante estadística descriptiva (tendencia central **y** dispersión).
4. Responder la pregunta de investigación utilizando agrupaciones (`groupby`) y 4 gráficos rotulados.
5. Redactar conclusiones fundamentadas en los datos junto con las **limitaciones** del estudio.

> ⚠️ Ejecute con **Entorno de ejecución → Reiniciar y ejecutar todo** antes de entregar.

# Parte 1 · Preparación y carga de datos

In [ ]:
# Instalar kagglehub si no está disponible en el entorno
%pip install -q kagglehub

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110
os.makedirs("figuras", exist_ok=True)
print("pandas:", pd.__version__)

### Cargar el archivo
Descargamos el dataset directamente desde Kaggle mediante `kagglehub` y ubicamos el archivo CSV automáticamente.

In [ ]:
path = kagglehub.dataset_download("larsen0966/student-performance-data-set")
print("Carpeta del dataset:", path)

archivos_csv = glob.glob(os.path.join(path, "*.csv"))
print("CSV encontrados:", archivos_csv)

df_original = pd.read_csv(archivos_csv[0])
df = df_original.copy()
print("Dimensiones (filas, columnas):", df.shape)
df.head()

---
# Parte 2 · Conocer los datos

**¿Qué representa una fila?** → Un **estudiante** de secundaria evaluado en tres periodos académicos (`G1`, `G2`, `G3`).  
**¿Qué es una variable?** → Un atributo demográfico, social, familiar o académico de ese estudiante.

In [ ]:
df.info()

In [ ]:
df[["age", "absences", "G1", "G2", "G3"]].describe().round(2)

### Diccionario de variables seleccionadas

Para mantener el análisis enfocado y comparable con el estudio de referencia, seleccionamos 5 factores clave y las 3 calificaciones de periodo:

| Columna original | Nombre en español | Qué contiene | Tipo |
|---|---|---|---|
| `studytime` | `tiempo_estudio` | Horas de estudio semanal (1: <2h, 2: 2–5h, 3: 5–10h, 4: >10h) | Categórica **ordinal** |
| `famsup` | `apoyo_familiar` | Apoyo educativo extra en casa (`yes` / `no`) | Categórica nominal |
| `internet` | `acceso_internet` | Acceso a internet en el hogar (`yes` / `no`) | Categórica nominal |
| `activities` | `extracurriculares` | Participación en actividades extracurriculares (`yes` / `no`) | Categórica nominal |
| `Medu` | `educacion_madre` | Nivel educativo de la madre (0: Ninguno a 4: Superior) | Categórica **ordinal** |
| `G1` | `nota_p1` | Calificación primer periodo (0–20) | Numérica continua |
| `G2` | `nota_p2` | Calificación segundo periodo (0–20) | Numérica continua |
| `G3` | `nota_final` | Calificación final del curso (0–20) | Numérica continua |

---
# Parte 3 · Calidad de los datos

Revisamos cinco dimensiones de calidad: **faltantes, duplicados, consistencia y mapeo de categorías, validez de rangos y valores atípicos**.

### 3.1 Valores faltantes y duplicados

In [ ]:
cols_interes = ["studytime", "famsup", "internet", "activities", "Medu", "G1", "G2", "G3"]
df = df[cols_interes].copy()

print("Celdas vacías por columna:")
print(df.isna().sum())
print("\nFilas duplicadas exactas (en todo el dataset original):", df_original.duplicated().sum())

### 3.2 Consistencia y traducción de categorías
**Decisión 1:** Renombrar columnas al español sin espacios ni mayúsculas.  
**Decisión 2:** Mapear códigos numéricos y en inglés (`yes`/`no`, niveles `0-4`, `1-4`) a etiquetas descriptivas en español y definir el orden lógico en variables ordinales.

In [ ]:
df = df.rename(columns={
    "studytime": "tiempo_estudio",
    "famsup": "apoyo_familiar",
    "internet": "acceso_internet",
    "activities": "extracurriculares",
    "Medu": "educacion_madre",
    "G1": "nota_p1",
    "G2": "nota_p2",
    "G3": "nota_final"
})

for col in ["apoyo_familiar", "acceso_internet", "extracurriculares"]:
    df[col] = df[col].str.strip().str.lower().map({"yes": "Sí", "no": "No"})

mapa_estudio = {
    1: "< 2 horas",
    2: "2 a 5 horas",
    3: "5 a 10 horas",
    4: "> 10 horas"
}
df["tiempo_estudio"] = df["tiempo_estudio"].map(mapa_estudio)

mapa_medu = {
    0: "Sin educación",
    1: "Primaria (4to)",
    2: "Básica (9no)",
    3: "Secundaria",
    4: "Educación Superior"
}
df["educacion_madre"] = df["educacion_madre"].map(mapa_medu)

# Verificar que no quedaron nulos tras el mapeo
assert df.notna().all().all(), "Existen valores sin mapear"

# Orden explícito para variables ordinales
df["tiempo_estudio"] = pd.Categorical(df["tiempo_estudio"], categories=list(mapa_estudio.values()), ordered=True)
df["educacion_madre"] = pd.Categorical(df["educacion_madre"], categories=list(mapa_medu.values()), ordered=True)

df.head()

### 3.3 Validez de rangos (escala 0 a 20)

In [ ]:
cols_nota = ["nota_p1", "nota_p2", "nota_final"]
print("Mínimo y máximo de cada calificación (escala válida 0 a 20):")
print(df[cols_nota].agg(["min", "max"]))

fuera = df[(df[cols_nota] < 0).any(axis=1) | (df[cols_nota] > 20).any(axis=1)]
print("\nFilas fuera de [0, 20]:", len(fuera))
print("Calificaciones iguales a 0:", (df[cols_nota] == 0).sum().to_dict())

### 3.4 Valores atípicos (regla del IQR)

In [ ]:
def detectar_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

for c in cols_nota:
    mask = detectar_iqr(df[c])
    print(f"{c:<12} atípicos: {mask.sum():>3}   valores: {sorted(df.loc[mask, c].tolist())}")

**Decisión 3 — Conservar los registros con nota 0 o atípicos:** En el sistema educativo portugués (escala 0–20), una nota de `0` en `nota_final` suele representar deserción del examen o reprobación total, no un error de digitación. Se conservan para no sesgar la muestra y se acompaña la media con la mediana.

### 3.5 Variables derivadas
- `promedio_periodos`: media simple de `nota_p1`, `nota_p2` y `nota_final`.
- `rendimiento`: categorización de `nota_final` según aprobación estándar en escala 0–20 (*Reprobado* < 10, *Aprobado* 10–13.9, *Sobresaliente* ≥ 14).

In [ ]:
df["promedio_periodos"] = df[cols_nota].mean(axis=1).round(2)
df["rendimiento"] = pd.cut(
    df["nota_final"],
    bins=[-0.1, 10, 14, 20.1],
    labels=["Reprobado (<10)", "Aprobado (10-13)", "Sobresaliente (14-20)"],
    right=False
)
df["rendimiento"].value_counts().sort_index()

---
# Parte 4 · Estadística descriptiva
Evaluamos tendencia central (media, mediana) junto con dispersión (desviación estándar, IQR y asimetría).

In [ ]:
resumen = df[cols_nota + ["promedio_periodos"]].agg(["count", "mean", "median", "std", "min", "max"]).T
resumen["iqr"] = df[cols_nota + ["promedio_periodos"]].quantile(0.75) - df[cols_nota + ["promedio_periodos"]].quantile(0.25)
resumen["asimetria"] = df[cols_nota + ["promedio_periodos"]].skew()
resumen.round(2)

### 4.1 Nota final (`nota_final`) según cada factor

In [ ]:
factores = ["tiempo_estudio", "apoyo_familiar", "acceso_internet", "extracurriculares", "educacion_madre"]

def tabla_por(factor):
    return (df.groupby(factor, observed=True)["nota_final"]
              .agg(n="count", media="mean", mediana="median", desv_std="std")
              .round(2))

for f in factores:
    print(f"\n=== Nota final según {f}")
    display(tabla_por(f))

### 4.2 Tamaño de la brecha en puntos y desviaciones estándar

In [ ]:
sd_general = df["nota_final"].std()
filas = []

for f in factores:
    medias = df.groupby(f, observed=True)["nota_final"].mean()
    mejor, peor = medias.idxmax(), medias.idxmin()
    brecha = medias.max() - medias.min()
    filas.append({
        "factor": f,
        "grupo_mayor": str(mejor),
        "grupo_menor": str(peor),
        "brecha_puntos": round(brecha, 2),
        "brecha_en_desv_std": round(brecha / sd_general, 2)
    })

brechas = pd.DataFrame(filas).sort_values("brecha_puntos", ascending=False).reset_index(drop=True)
brechas

### 4.3 Distribución de aprobación por tiempo de estudio y acceso a internet

In [ ]:
for f in ["tiempo_estudio", "acceso_internet"]:
    print(f"\n=== {f} vs rendimiento (% por fila)")
    display((pd.crosstab(df[f], df["rendimiento"], normalize="index") * 100).round(1))

### 4.4 Correlación entre los tres periodos académicos

In [ ]:
corr = df[cols_nota].corr().round(2)
corr

---
# Parte 5 · Gráficos
Cada visualización responde una pregunta analítica específica, incluye rótulos claros y se exporta en alta resolución a `figuras/`.

### Gráfico 1 · ¿Cómo se distribuyen las calificaciones por periodo?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
colores = ["#2b5c8f", "#2ca02c", "#d95f02"]
titulos_p = ["Primer Periodo (G1)", "Segundo Periodo (G2)", "Nota Final (G3)"]

for ax, c, col, tit in zip(axes, cols_nota, colores, titulos_p):
    ax.hist(df[c], bins=15, color=col, edgecolor="white", alpha=0.9)
    ax.axvline(df[c].mean(), color="black", linestyle="--", linewidth=1.2, label=f"Media {df[c].mean():.1f}")
    ax.axvline(df[c].median(), color="gray", linestyle=":", linewidth=1.6, label=f"Mediana {df[c].median():.1f}")
    ax.set_title(tit)
    ax.set_xlabel("Calificación (0-20)")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Número de estudiantes")
fig.suptitle(f"Distribución de calificaciones por periodo (n = {len(df)})", y=1.03)
plt.tight_layout()
plt.savefig("figuras/g1_distribucion_periodos.png", bbox_inches="tight")
plt.show()

**Lectura:** Las calificaciones se concentran alrededor de 11–12 puntos en los tres periodos. Sin embargo, en la Nota Final (G3) aparece una acumulación notable en `0` (estudiantes reprobados o retirados al cierre), lo que desplaza ligeramente la media por debajo de la mediana.

### Gráfico 2 · ¿Varía la nota final según acceso a internet y apoyo familiar?

In [ ]:
def caja_por(ax, factor, titulo):
    grupos = [g for g in df[factor].dropna().unique()]
    datos = [df.loc[df[factor] == g, "nota_final"] for g in grupos]
    bp = ax.boxplot(
        datos,
        tick_labels=[f"{g}\n(n={len(d)})" for g, d in zip(grupos, datos)],
        patch_artist=True,
        showmeans=True
    )
    for patch, col in zip(bp["boxes"], ["#a6bddb", "#fdbb84"]):
        patch.set_facecolor(col)
    ax.set_title(titulo)
    ax.set_ylabel("Nota Final (0-20)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
caja_por(axes[0], "acceso_internet", "Según acceso a internet en casa")
caja_por(axes[1], "apoyo_familiar", "Según apoyo educativo familiar")
fig.suptitle("Distribución de la Nota Final por condiciones del hogar (triángulo = media)", y=1.02)
plt.tight_layout()
plt.savefig("figuras/g2_cajas_hogar.png", bbox_inches="tight")
plt.show()

**Lectura:** Quienes cuentan con acceso a internet presentan una mediana superior (12.0 vs 11.0) y un tercer cuartil más alto. Por el contrario, el apoyo familiar extra muestra medianas prácticamente idénticas entre ambos grupos, lo que sugiere que dicho apoyo suele brindarse de manera reactiva a estudiantes con dificultades previas.

### Gráfico 3 · ¿Aumenta la nota final con la educación materna y el tiempo de estudio?

In [ ]:
t = (df.groupby("educacion_madre", observed=True)["nota_final"]
       .agg(["mean", "count"]))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
barras = ax.barh(t.index.astype(str), t["mean"], color="#41b6c4")
for b, (m, n) in zip(barras, zip(t["mean"], t["count"])):
    ax.text(m + 0.15, b.get_y() + b.get_height()/2, f"{m:.2f}  (n={n})", va="center", fontsize=9)

ax.axvline(df["nota_final"].mean(), color="black", linestyle="--", linewidth=1,
           label=f"Media general {df['nota_final'].mean():.2f}")
ax.set_xlim(0, t["mean"].max() + 2.5)
ax.set_xlabel("Promedio Nota Final (0-20)")
ax.set_ylabel("Nivel educativo de la madre")
ax.set_title("Nota final promedio según el nivel educativo de la madre")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("figuras/g3_educacion_madre.png", bbox_inches="tight")
plt.show()

**Lectura:** Descartando el grupo *Sin educación* (que cuenta con apenas n=3 observaciones y alta variabilidad muestral), se observa una tendencia ascendente progresiva desde *Primaria* hasta *Educación Superior*, donde los estudiantes superan en más de 1 punto la media general.

### Gráfico 4 · ¿Qué tan fuerte es la continuidad entre periodos académicos?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.3, 1]})

ax = axes[0]
for g, col in [("Sí", "#1b9e77"), ("No", "#d95f02")]:
    sub = df[df["acceso_internet"] == g]
    ax.scatter(sub["nota_p2"], sub["nota_final"], s=18, alpha=0.6, color=col, label=f"Internet: {g}")

r = df["nota_p2"].corr(df["nota_final"])
ax.set_xlabel("Calificación Segundo Periodo (G2)")
ax.set_ylabel("Calificación Final (G3)")
ax.set_title(f"Segundo Periodo vs. Nota Final (r = {r:.2f})")
ax.legend()

ax = axes[1]
im = ax.imshow(corr.values, cmap="YlGnBu", vmin=0.7, vmax=1.0)
ax.set_xticks(range(3)); ax.set_xticklabels(["P1", "P2", "Final"])
ax.set_yticks(range(3)); ax.set_yticklabels(["P1", "P2", "Final"])
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                color="white" if corr.values[i, j] > 0.88 else "black")
ax.set_title("Matriz de correlación entre periodos")
fig.colorbar(im, ax=ax, label="r de Pearson", shrink=0.8)

plt.tight_layout()
plt.savefig("figuras/g4_correlacion_periodos.png", bbox_inches="tight")
plt.show()

**Lectura:** Existe una correlación lineal muy fuerte entre el segundo periodo y la nota final ($r = 0.92$). Los puntos situados sobre el eje horizontal ($G3 = 0$ con $G2 > 0$) corresponden a estudiantes que no presentaron la evaluación final o desertaron al cierre del curso.

---
# Parte 6 · Conclusiones y Limitaciones

In [ ]:
print(f"Muestra total: {len(df)} estudiantes. Nota final promedio = {df['nota_final'].mean():.2f} "
      f"± {df['nota_final'].std():.2f} (mediana {df['nota_final'].median():.2f}).\n")
for _, r in brechas.iterrows():
    print(f"- {r['factor']}: '{r['grupo_mayor']}' supera a '{r['grupo_menor']}' por "
          f"{r['brecha_puntos']} puntos ({r['brecha_en_desv_std']} desv. std.).")
print(f"\nCorrelación entre periodos: min r = {corr.values[np.triu_indices(3, 1)].min():.2f}, "
      f"max r = {corr.values[np.triu_indices(3, 1)].max():.2f}.")

### Conclusiones del análisis

1. **Factores socioeducativos con mayor brecha:** El nivel educativo de la madre (`educacion_madre`), el tiempo dedicado al estudio semanal (`tiempo_estudio`) y el acceso a internet en el hogar (`acceso_internet`) presentan las asociaciones positivas más claras con una mayor calificación final.
2. **Compromiso de estudio:** Quienes estudian más de 10 horas semanales obtienen en promedio alrededor de 1.3 puntos más (en escala 0–20) frente a quienes estudian menos de 2 horas.
3. **Continuidad académica:** El desempeño en los periodos previos (`G1` y `G2`) mantiene una correlación superior a $0.85$ con el resultado final, consolidándose como el indicador de seguimiento más consistente.

### Limitaciones
* **Asociación, no causalidad:** Al tratarse de datos observacionales de corte transversal, no es posible afirmar que el acceso a internet o el tiempo de estudio causen por sí solos el aumento de la nota sin controlar otras variables socioeconómicas.
* **Contexto geográfico y temporal:** El conjunto de datos proviene de dos escuelas secundarias en Portugal (datos históricos UCI), por lo que sus distribuciones y dinámicas escolares no son directamente extrapolables al sistema educativo colombiano actual.
* **Subgrupos con tamaño reducido:** Categorías como *Sin educación* en `educacion_madre` (n=3) tienen muy pocas observaciones, por lo que su promedio presenta alta incertidumbre muestral.
* **Efecto de ceros en la nota final:** La presencia de notas iguales a `0` al cierre del curso influye en la media y la desviación estándar; en un modelo predictivo futuro convendría analizar por separado la deserción frente al rendimiento numérico.